# SAXSess-I: absolute-intensity N008 reference data

This notebook converts the compact 2026-06-08 SAXSess-I acquisition set, prepares the full correction graph, and processes all six N008 aliquots into absolute `I(Q)` curves. It targets MoDaCor 1.8.0 and writes only below `work/`.

The manual instrument needs four foil-fluorescence acquisitions in addition to the N008, water, empty-capillary, and empty-chamber scattering measurements. Preprocessing reconstructs the chained transmission and relative flux; MoDaCor performs detector/time/flux/transmission normalization, water subtraction, geometry and solid-angle correction, thickness normalization, water-based absolute calibration, azimuthal averaging, and uncertainty propagation.

In [ ]:
from pathlib import Path
import atexit
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("BAM/SAXSess_I")
sys.path.insert(0, str(PROJECT_DIR))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from modacor.client import LocalRuntimeServer, RuntimeClient
from modacor.runner.pipeline import Pipeline

from saxsess_helpers import (
    build_n008_plan,
    load_metadata,
    preprocess_measurements,
    read_absolute_csv,
    validate_source_files,
)

## Configuration

Leave `RUNTIME_URL` as `None` for a notebook-owned local server. To switch to an existing server, set its URL and, when needed, the server-visible path corresponding to `PROJECT_DIR`. Set `SAMPLE_IDS` to a list such as `["S00635"]` for a shorter smoke run.

In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "20260608_N008"
WORKBOOK = DATA_DIR / "metadata_N008.xlsx"
WATER_REFERENCE = PROJECT_DIR / "data" / "calibration" / "water_cal_reference.pdh"
STATIC_CONFIG = PROJECT_DIR / "config" / "SAXSess_I.yaml"
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "SAXSess_I_N008_absolute.yaml"
WORK_DIR = PROJECT_DIR / "work"
PREPROCESSED_DIR = WORK_DIR / "preprocessed"
OUTPUT_DIR = WORK_DIR / "output"

SAMPLE_IDS = None
RUNTIME_URL = None
RUNTIME_PROJECT_DIR = None
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8902
OUTPUT_DATA_PATHS = ["/sample/Q", "/sample/signal"]

## Inventory and preprocess the compact acquisition set

In [ ]:
metadata = load_metadata(WORKBOOK)
validate_source_files(metadata, DATA_DIR)
plan = build_n008_plan(metadata)
if SAMPLE_IDS is not None:
    plan = plan[plan["measurement_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
    missing = sorted(set(SAMPLE_IDS).difference(plan["measurement_id"]))
    if missing:
        raise ValueError(f"Unknown N008 SAMPLE_IDS: {missing}")

print(f"Measurements: {len(metadata)} scattering files; N008 outputs: {len(plan)}")
display(plan)

In [ ]:
preprocessed = preprocess_measurements(
    metadata,
    raw_dir=DATA_DIR,
    static_config=STATIC_CONFIG,
    output_dir=PREPROCESSED_DIR,
)
print(f"Wrote {len(preprocessed)} HDF5 sources to {PREPROCESSED_DIR.relative_to(PROJECT_DIR)}.")

## Prepare and inspect the correction graph

In [ ]:
pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
display(Markdown(f"```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
print(f"Prepared {len(pipeline.graph)} steps from {PIPELINE_PATH.name}.")

## Start the local runtime or connect to an existing one

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = None
if RUNTIME_URL is None:
    server = LocalRuntimeServer(
        host=SERVER_HOST,
        port=SERVER_PORT,
        log_path=OUTPUT_DIR / "modacor_server.log",
    )
    client = server.start()
    atexit.register(server.stop)
    print(f"{'Started' if server.launched else 'Reused'} runtime at {client.base_url}")
else:
    client = RuntimeClient(RUNTIME_URL, timeout=300.0)
    client.wait_until_ready()
    print(f"Connected to external runtime at {client.base_url}")

## Process all selected N008 aliquots

One runtime session is reused. Only the sample source changes after the first run; the water/background/calibration branches remain cached.

In [ ]:
def runtime_path(path):
    path = Path(path).resolve()
    if RUNTIME_PROJECT_DIR is None:
        return str(path)
    return (Path(RUNTIME_PROJECT_DIR) / path.relative_to(PROJECT_DIR)).as_posix()

if plan.empty:
    raise ValueError("No N008 measurements selected.")
for column in ("sample_background_id", "intensity_calibration_id", "intensity_calibration_background_id"):
    if plan[column].nunique() != 1:
        raise ValueError(f"Selected measurements do not share one {column}.")
fixed = plan.iloc[0]
session = client.replace_session(
    "saxsess-i-n008-absolute",
    name="SAXSess-I N008 absolute-intensity batch",
    pipeline_yaml=PIPELINE_PATH.read_text(encoding="utf-8"),
    trace={"enabled": False},
)
session.register_sources(
    {"ref": "sample_background", "type": "hdf", "location": runtime_path(preprocessed[fixed["sample_background_id"]])},
    {"ref": "intensity_calibration", "type": "hdf", "location": runtime_path(preprocessed[fixed["intensity_calibration_id"]])},
    {"ref": "intensity_calibration_background", "type": "hdf", "location": runtime_path(preprocessed[fixed["intensity_calibration_background_id"]])},
    {"ref": "defaults", "type": "yaml", "location": runtime_path(STATIC_CONFIG)},
    {
        "ref": "i_cal_reference",
        "type": "csv",
        "location": runtime_path(WATER_REFERENCE),
        "kwargs": {
            "iosource_method_kwargs": {
                "skip_header": 5,
                "max_rows": 725,
                "usecols": [0, 1, 2],
                "names": ["Q", "I", "I_sigma"],
                "dtype": "float",
                "encoding": "utf-8",
            }
        },
    },
)

results = {}
for row in plan.itertuples(index=False):
    csv_path = OUTPUT_DIR / f"{row.measurement_id}_absolute.csv"
    hdf_path = OUTPUT_DIR / f"{row.measurement_id}_absolute.h5"
    session.register_source(
        {"ref": "sample", "type": "hdf", "location": runtime_path(preprocessed[row.measurement_id])}
    )
    session.register_sink(
        {
            "ref": "export_csv",
            "type": "csv",
            "location": runtime_path(csv_path),
            "kwargs": {"iosink_method_kwargs": {"delimiter": ";"}},
        }
    )
    result = session.process(
        mode="auto",
        changed_sources=["sample"],
        run_name=row.measurement_id,
        rollback_snapshot=False,
        write_hdf={"path": runtime_path(hdf_path), "data_paths": OUTPUT_DATA_PATHS},
    )
    results[row.measurement_id] = {"csv": csv_path, "hdf": hdf_path, "run": result}
    print(f"{row.measurement_id}: {result['status']} ({result['effective_mode']})")

## Verify units and compare the six absolute curves

In [ ]:
figure, axis = plt.subplots(figsize=(9, 6))
for measurement_id, item in results.items():
    unit_row = item["csv"].read_text(encoding="utf-8").splitlines()[1].split(";")
    if unit_row != ["1/nm", "1/nm", "1/m/sr", "1/m/sr"]:
        raise ValueError(f"Unexpected output units for {measurement_id}: {unit_row}")
    curve = read_absolute_csv(item["csv"])
    valid = np.isfinite(curve["Q"]) & np.isfinite(curve["I"]) & (curve["Q"] > 0) & (curve["I"] > 0)
    uncertainty_valid = valid & np.isfinite(curve["Q_sigma"]) & np.isfinite(curve["I_sigma"]) & (curve["Q_sigma"] >= 0) & (curve["I_sigma"] >= 0)
    (line,) = axis.plot(curve.loc[valid, "Q"], curve.loc[valid, "I"], marker=".", label=measurement_id)
    axis.errorbar(
        curve.loc[uncertainty_valid, "Q"],
        curve.loc[uncertainty_valid, "I"],
        xerr=curve.loc[uncertainty_valid, "Q_sigma"],
        yerr=curve.loc[uncertainty_valid, "I_sigma"],
        fmt="none",
        ecolor=line.get_color(),
        elinewidth=0.7,
        alpha=0.35,
    )
axis.set(xscale="log", yscale="log", xlabel=r"$q$ (nm$^{-1}$)", ylabel=r"$I$ (m$^{-1}$ sr$^{-1}$)")
axis.grid(True, which="both", alpha=0.25)
axis.legend(title="N008 aliquot")
axis.set_title(r"SAXSess-I N008 absolute intensity with $1\sigma$ uncertainties")
figure.tight_layout()

## Cleanup

This stops only a process launched by the notebook. An external runtime is left untouched.

In [ ]:
if server is not None:
    server.stop()
    print("Stopped the notebook-owned runtime.")
else:
    print("Left the external runtime running.")